In [12]:
%pip install pydantic-settings

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [2]:
import asyncio
import json
import os
import time
from pathlib import Path
# Ensure project parent directory is on sys.path so `settings` can be imported from the project root
import sys
sys.path.insert(0, str(Path('..').resolve()))
from settings import my_settings
import pandas as pd
from openai import AsyncOpenAI
import os, sys
from dotenv import load_dotenv

load_dotenv()

# Make sure your OPEN_AI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPEN_AI_API_KEY first'

#client = AsyncOpenAI()

client = AsyncOpenAI(
    api_key=my_settings.OPENAI_API_KEY,
    base_url=my_settings.OPENAI_BASE_URL
)

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [3]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [4]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot.  no examples, no persona."""
    # Done: return a messages list like [{'role': 'user', 'content': '...'}]
    content = (
        "Extract these fields from the job snippet and return ONLY a JSON object:"
        "  - company: string or null"
        "  - role: string or null"
        "  - years_experience_required: integer or null"
        "If the snippet doesn't specify a value, return null for that field."
        f"Job snippet:\n{snippet_text}\nReturn the JSON only."
    )
    return [{"role": "user", "content": content}]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    # Done: same shape, but include examples
       
    system = "You are a precise extractor that outputs a single JSON object with keys company, role, years_experience_required."

    ex1_user = "Example snippet: Acme Corp is hiring a Senior ML Engineer — 5+ years experience required.Return the JSON."
    ex1_assistant = '{"company": "Acme Corp", "role": "Senior ML Engineer", "years_experience_required": 5}'

    ex2_user = "Example snippet: We seek a backend developer (no specific years listed). Return the JSON."
    ex2_assistant = '{"company": null, "role": "Backend Developer", "years_experience_required": null}'

    final_user = f"Job snippet:\n{snippet_text}\nReturn the JSON only."

    return [
        {"role": "system", "content": system},
        {"role": "user", "content": ex1_user},
        {"role": "assistant", "content": ex1_assistant},
        {"role": "user", "content": ex2_user},
        {"role": "assistant", "content": ex2_assistant},
        {"role": "user", "content": final_user},
    ]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    # Done: 'You are an expert recruiter... Output JSON with these exact fields...'
    
    system = (
        "You are an expert data extractor. Output MUST be a single JSON object with exactly these keys:\n"
        "  - company: string or null"
        "  - role: string or null"
        "  - years_experience_required: integer or null"
        "Do not add any commentary, extra fields, or explanatory text. If you see ranges like '3-5 years', use the minimum (3). If you see '5+ years', use 5."
    )
    user = f"Job snippet:\n{snippet_text}\nProduce the JSON object exactly as specified."
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    # Done: 'Think step by step, then answer with JSON.'
    
    system = (
        "You may provide 1-3 brief reasoning lines prefixed with 'Reasoning:' then on a new line output only the final JSON object."
        " The JSON must contain keys company, role, years_experience_required (integer or null)."
    )
    user = f"Job snippet:\n{snippet_text}\nThink step by step, then give a short reasoning (1-3 lines starting with 'Reasoning:'), then on a new line output only the JSON."
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}


## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [5]:
import re
import json
import time


def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    # Done: extract + parse the JSON, return a dict or None
    
    if not text:
        return None
    # Strip ``` fences and language hints
    text = re.sub(r"```(?:json)?\n", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n```", "", text)
    # Find first '{' and last '}' and attempt to parse
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        return None
    candidate = text[start:end+1]
    try:
        return json.loads(candidate)
    except Exception:
        # Last resort: try to extract key:value pairs via heuristic
        return None


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    # Done: call the model, time the call, compute cost, parse the response
    
    # accept different snippet shapes (some have 'snippet' key)
    snippet_text = snippet.get('text') or snippet.get('snippet') or snippet.get('job') or snippet.get('description') or snippet.get('content') or ''
    messages = STRATEGIES[strategy_name](snippet_text)
    start = time.perf_counter()
    # call the AsyncOpenAI client
    # use retries similar to utils.call_open_ai 
    for attempt in range(getattr(my_settings, 'OPEN_AI_RETRIES', 1)):
        try:
            resp = await client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=TEMPERATURE,
                timeout=getattr(my_settings, 'OPEN_AI_TIMEOUT', 60)
            )
            end = time.perf_counter()
            latency = end - start
            # extract text from response (new client: choices[0].message.content)
            raw = resp.choices[0].message.content
            parsed = parse_response(raw)
            # token counts if available
            prompt_tokens = getattr(resp.usage, 'prompt_tokens', None)
            completion_tokens = getattr(resp.usage, 'completion_tokens', None)
            total_tokens = getattr(resp.usage, 'total_tokens', None)
            # estimate cost using RATES if available
            rates = RATES.get(MODEL, None)
            if rates and prompt_tokens is not None and completion_tokens is not None:
                cost = prompt_tokens * rates['in'] + completion_tokens * rates['out']
            else:
                cost = 0.0
            return {
                'strategy': strategy_name,
                'snippet_id': snippet.get('id'),
                'raw_response': raw,
                'parsed': parsed,
                'cost_usd': cost,
                'latency_s': latency,
                'model': getattr(resp, 'model', MODEL),
                'prompt_tokens': prompt_tokens,
                'completion_tokens': completion_tokens,
                'total_tokens': total_tokens,
            }
        except Exception as e:
            if attempt == getattr(my_settings, 'OPEN_AI_RETRIES', 1) - 1:
                return {
                    'strategy': strategy_name,
                    'snippet_id': snippet.get('id'),
                    'raw_response': None,
                    'parsed': None,
                    'cost_usd': 0.0,
                    'latency_s': time.perf_counter() - start,
                    'model': MODEL,
                    'error': str(e),
                }
            await asyncio.sleep(1)


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    # Done: build the task list, gather, return results
    
    tasks = []
    for strategy_name in STRATEGIES.keys():
        for snippet in snippets:
            tasks.append(run_one(strategy_name, snippet))
    results = await asyncio.gather(*tasks)
    return results


In [7]:
# HTTPX-based async runner 
import httpx
import os
import asyncio

MAX_CONCURRENCY = getattr(my_settings, 'OPEN_AI_MAX_CONCURRENCY', 8)
RETRIES = getattr(my_settings, 'OPEN_AI_RETRIES', 3)
HTTPX_SEMAPHORE = asyncio.Semaphore(MAX_CONCURRENCY)
base = my_settings.OPENAI_BASE_URL.rstrip('/')
if '/v1' in base:
    ENDPOINT = base + '/chat/completions'
else:
    ENDPOINT = base + '/v1/chat/completions'

HEADERS = {
    'Authorization': f"Bearer {os.environ.get('OPENAI_API_KEY')}",
    'Content-Type': 'application/json'
}


async def _httpx_post(body: dict) -> dict:
    async with httpx.AsyncClient(timeout=60.0) as client:
        for attempt in range(RETRIES):
            try:
                resp = await client.post(ENDPOINT, headers=HEADERS, json=body)
                resp.raise_for_status()
                return resp.json()
            except Exception as e:
                if attempt == RETRIES - 1:
                    raise
                await asyncio.sleep(1 + attempt)


async def run_one_httpx(strategy_name: str, snippet: dict) -> dict:
    snippet_text = snippet.get('text') or snippet.get('snippet') or snippet.get('job') or snippet.get('description') or snippet.get('content') or ''
    messages = STRATEGIES[strategy_name](snippet_text)
    body = {"model": MODEL, "messages": messages, "temperature": TEMPERATURE}

    start = time.perf_counter()
    try:
        async with HTTPX_SEMAPHORE:
            data = await _httpx_post(body)
    except Exception as e:
        return {
            'strategy': strategy_name,
            'snippet_id': snippet.get('id'),
            'raw_response': None,
            'parsed': None,
            'cost_usd': 0.0,
            'latency_s': time.perf_counter() - start,
            'model': MODEL,
            'error': str(e),
        }

    end = time.perf_counter()
    latency = end - start

    choice = (data.get('choices') or [{}])[0]
    message = choice.get('message') or choice.get('delta') or {}
    raw = message.get('content') if isinstance(message, dict) else None
    if raw is None and isinstance(choice, dict):
        raw = choice.get('text') or json.dumps(choice)

    parsed = parse_response(raw)
    usage = data.get('usage', {}) or {}
    prompt_tokens = usage.get('prompt_tokens')
    completion_tokens = usage.get('completion_tokens')
    total_tokens = usage.get('total_tokens')

    rates = RATES.get(MODEL, None)
    if rates and prompt_tokens is not None and completion_tokens is not None:
        cost = prompt_tokens * rates['in'] + completion_tokens * rates['out']
    else:
        cost = 0.0

    return {
        'strategy': strategy_name,
        'snippet_id': snippet.get('id'),
        'raw_response': raw,
        'parsed': parsed,
        'cost_usd': cost,
        'latency_s': latency,
        'model': data.get('model', MODEL),
        'prompt_tokens': prompt_tokens,
        'completion_tokens': completion_tokens,
        'total_tokens': total_tokens,
    }


async def run_all_httpx() -> list[dict]:
    """Run all calls using the httpx runner with concurrency control."""
    tasks = [run_one_httpx(sname, sn) for sname in STRATEGIES.keys() for sn in snippets]
    results = await asyncio.gather(*tasks)
    return results


In [8]:
# Run full httpx batch (40 calls) and save results
print('Running full httpx batch (40 calls) using httpx runner...')
results_httpx = await run_all_httpx()
print(f'Got {len(results_httpx)} results.')
import json
with open('mp1_results_httpx.jsonl', 'w', encoding='utf8') as f:
    for r in results_httpx:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print('Saved results to mp1_results_httpx.jsonl')
import pandas as pd
df_httpx = pd.DataFrame(results_httpx)
df_httpx.head()
results_httpx

Running full httpx batch (40 calls) using httpx runner...
Got 40 results.
Saved results to mp1_results_httpx.jsonl


[{'strategy': 'zero_shot',
  'snippet_id': 'j01',
  'raw_response': '```json\n{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5\n}\n```',
  'parsed': {'company': 'Acme Corp',
   'role': 'Senior Software Engineer',
   'years_experience_required': 5},
  'cost_usd': 3.57e-05,
  'latency_s': 4.307667900000524,
  'model': 'gpt-4o-mini-2024-07-18',
  'prompt_tokens': 102,
  'completion_tokens': 34,
  'total_tokens': 136},
 {'strategy': 'zero_shot',
  'snippet_id': 'j02',
  'raw_response': '```json\n{\n  "company": "Northwind Ltd.",\n  "role": "Data Analyst",\n  "years_experience_required": 2\n}\n```',
  'parsed': {'company': 'Northwind Ltd.',
   'role': 'Data Analyst',
   'years_experience_required': 2},
  'cost_usd': 3.51e-05,
  'latency_s': 3.89983840000059,
  'model': 'gpt-4o-mini-2024-07-18',
  'prompt_tokens': 102,
  'completion_tokens': 33,
  'total_tokens': 135},
 {'strategy': 'zero_shot',
  'snippet_id': 'j03',
  'raw_response': '``

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [9]:
def _norm_str(s: str) -> str:
    if s is None:
        return ''
    return ' '.join(str(s).strip().lower().split())


def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0..3."""
    if not extracted:
        return 0
    score = 0
    # company
    if _norm_str(extracted.get('company')) and _norm_str(gold.get('company')):
        if _norm_str(extracted.get('company')) == _norm_str(gold.get('company')):
            score += 1
    else:
        # treat both empty as match
        if not _norm_str(extracted.get('company')) and not _norm_str(gold.get('company')):
            score += 1
    # role
    if _norm_str(extracted.get('role')) and _norm_str(gold.get('role')):
        if _norm_str(extracted.get('role')) == _norm_str(gold.get('role')):
            score += 1
    else:
        if not _norm_str(extracted.get('role')) and not _norm_str(gold.get('role')):
            score += 1
    # years_experience_required (numeric or null)
    ext_years = extracted.get('years_experience_required') if extracted is not None else None
    gold_years = gold.get('years_experience_required')
    if ext_years is None and gold_years is None:
        score += 1
    else:
        try:
            if ext_years is not None and int(ext_years) == int(gold_years):
                score += 1
        except Exception:
            pass
    return score


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-25 (higher=better).

    We'll prompt the judge with the snippet, the gold JSON, and the extracted JSON and ask for a 1-25 score and a short justification.
    """
    # build judge prompt
    prompt = (
        "You are an expert evaluator. Given a job snippet, the reference (gold) extraction, and a model's extracted JSON,\n"
        "score how well the model did on a scale from 1 to 25 (25 = perfect).\n"
        "Return only a JSON object with keys: 'score' (int 1-25) and 'reason' (short string).\n\n"
        f"Job snippet:\n{snippet_text}\n\n"
        f"Gold extraction:\n{json.dumps(gold, ensure_ascii=False, indent=2)}\n\n"
        f"Model extraction:\n{json.dumps(extracted, ensure_ascii=False, indent=2)}\n\n"
        "Provide the JSON only."
    )
    # call judge model
    for attempt in range(getattr(my_settings, 'OPEN_AI_RETRIES', 1)):
        try:
            resp = await client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                timeout=getattr(my_settings, 'OPEN_AI_TIMEOUT', 60)
            )
            text = resp.choices[0].message.content
            parsed = parse_response(text)
            if not parsed:
                # try to parse as plain number
                try:
                    val = int(text.strip())
                    return max(1, min(25, val))
                except Exception:
                    return 1
            score = parsed.get('score') or parsed.get('score') == 0 and 0
            try:
                score = int(score)
            except Exception:
                score = 1
            return max(1, min(25, score))
        except Exception as e:
            if attempt == getattr(my_settings, 'OPEN_AI_RETRIES', 1) - 1:
                return 1
            await asyncio.sleep(1)


In [10]:
# Apply scoring to the httpx results
print('Scoring httpx results...')
scored = []
for row in results_httpx:
    snip_id = row['snippet_id']
    snippet = next((s for s in snippets if s.get('id') == snip_id), None)
    snippet_text = snippet.get('snippet') if snippet else ''
    extracted = row.get('parsed')
    gold_row = golden.get(snip_id)
    accuracy = score_accuracy(extracted, gold_row)
    parse_success = extracted is not None
    # call judge model
    judge_score = await score_llm_judge(snippet_text, extracted, gold_row)
    out = dict(row)
    out.update({'accuracy': accuracy, 'parse_success': parse_success, 'llm_judge_score': judge_score})
    scored.append(out)

# save
import json
with open('mp1_scored_httpx.jsonl', 'w', encoding='utf8') as f:
    for r in scored:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print(f'Scored {len(scored)} results and saved to mp1_scored_httpx.jsonl')


Scoring httpx results...
Scored 40 results and saved to mp1_scored_httpx.jsonl


## Step 5 — Build the comparison table

In [12]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(3)

summary.columns = ['Accuracy (mean)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

,Accuracy (mean),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.8,1.0,23.1,0.001,11.628
few_shot,3.0,1.0,23.6,0.000,5.632
structured,2.9,1.0,23.7,0.000,9.121
zero_shot,2.6,1.0,23.3,0.000,3.897


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```